# Bay Area Households by County (ACS 1-Year Estimates)

Queries the Census Bureau ACS API for variable **B11001_001E** (total households)
across the 9 Bay Area counties for the last 10 years (2015–2024).

In [1]:
from census import Census
import pandas as pd

# Load Census API key
with open(r"M:\Data\Census\API\api-key.txt") as f:
    CENSUS_API_KEY = f.read().strip()

c = Census(CENSUS_API_KEY)

BAY_AREA_COUNTIES = {
    "001": "Alameda",
    "013": "Contra Costa",
    "041": "Marin",
    "055": "Napa",
    "075": "San Francisco",
    "081": "San Mateo",
    "085": "Santa Clara",
    "095": "Solano",
    "097": "Sonoma",
}

# 2020 ACS 1-year was not released due to COVID data quality issues
YEARS = [y for y in range(2015, 2025) if y != 2020]

VARIABLES = {
    "B11001_001E": "households",         # B11001: Household Type — total households (estimate)
    "B11001_001M": "households_moe",     # B11001: Household Type — total households (margin of error)
    "B01003_001E": "population",         # B01003: Total Population (estimate)
    # B01003 MOE is not published by Census (controlled estimate); API returns -555555555
    "B25008_001E": "hh_population",      # B25008: Total Population in Occupied Housing Units (estimate)
    "B25008_001M": "hh_population_moe",  # B25008: Total Population in Occupied Housing Units (margin of error)
}

In [2]:
records = []

county_list = ",".join(BAY_AREA_COUNTIES.keys())

for year in YEARS:
    rows = c.acs1.get(
        ("NAME", *VARIABLES.keys()),
        {"for": f"county:{county_list}", "in": "state:06"},
        year=year,
    )
    for row in rows:
        county_fips = row["county"]
        if county_fips in BAY_AREA_COUNTIES:
            record = {"year": year, "county": BAY_AREA_COUNTIES[county_fips]}
            for api_var, col_name in VARIABLES.items():
                record[col_name] = int(row[api_var])
            records.append(record)

df = pd.DataFrame(records).sort_values(["year", "county"]).reset_index(drop=True)

In [3]:
import numpy as np

def rss(x):
    """Root-sum-of-squares for combining margins of error."""
    return int(round(np.sqrt((x.astype(float) ** 2).sum())))

# Compute Bay Area totals: sum estimates, root-sum-of-squares for MOEs
bay_rows = (
    df.groupby("year")
    .agg(
        households=("households", "sum"),
        households_moe=("households_moe", rss),
        population=("population", "sum"),
        hh_population=("hh_population", "sum"),
        hh_population_moe=("hh_population_moe", rss),
    )
    .reset_index()
    .assign(county="Bay Area")
)

df_all = pd.concat([df, bay_rows], ignore_index=True)

county_order = sorted(BAY_AREA_COUNTIES.values()) + ["Bay Area"]

def make_wide_table(data, estimate_col, moe_col=None):
    """Pivot to wide format. If moe_col is given, interleave MOE after each county estimate."""
    cols = [estimate_col] + ([moe_col] if moe_col else [])
    pivot = data.pivot(index="year", columns="county", values=cols)
    col_tuples = [(col, county) for county in county_order for col in cols]
    pivot = pivot[col_tuples]
    pivot.columns = [
        county if col == estimate_col else f"{county} MOE"
        for col, county in col_tuples
    ]
    pivot.index.name = "Year"
    return pivot

hh_table     = make_wide_table(df_all, "households",    "households_moe")
pop_table    = make_wide_table(df_all, "population")    # MOE not published for B01003
hh_pop_table = make_wide_table(df_all, "hh_population", "hh_population_moe")

print("Households (B11001_001E / B11001_001M)")
display(hh_table)
print("\nTotal Population (B01003_001E) — MOE not available for controlled estimates")
display(pop_table)
print("\nPopulation in Occupied Housing Units (B25008_001E / B25008_001M)")
display(hh_pop_table)

Households (B11001_001E / B11001_001M)


,Alameda,Alameda MOE,Contra Costa,Contra Costa MOE,Marin,Marin MOE,Napa,Napa MOE,San Francisco,San Francisco MOE,San Mateo,San Mateo MOE,Santa Clara,Santa Clara MOE,Solano,Solano MOE,Sonoma,Sonoma MOE,Bay Area,Bay Area MOE
Year,,,,,,,,,,,,,,,,,,,,
2015,571828,2806,391996,2551,105887,1549,49619,1123,356916,3701,263280,1811,633786,3221,145502,1849,190662,2442,2709476,7401
2016,572012,3657,391288,2968,106333,1579,49561,1328,358703,4561,263445,2321,635687,3094,149172,1723,187504,2620,2713705,8487
2017,573589,3461,392046,2369,104219,2062,47848,1322,360323,4331,264185,2424,633811,3358,151127,1649,188829,2691,2715977,8334
2018,575410,4162,396133,3132,104954,1838,47315,1405,362827,5010,259654,3087,645108,4162,152291,1537,187434,2355,2731126,9610
2019,585632,3709,399792,2910,105298,1492,48107,1441,365851,4759,265003,2718,643637,4477,150393,1826,190689,2478,2754402,9278
2021,589180,4004,411560,2396,103378,1853,49979,1257,350796,5380,264135,2347,650593,3591,157617,1748,190586,2185,2767824,9053
2022,596614,4260,415194,2740,103285,1968,51025,1521,361912,4951,262122,3454,656477,4082,159294,1469,194698,2077,2800621,9561
2023,608534,4026,416172,2488,101608,2158,51167,1100,372027,4832,265124,3090,665549,3723,157766,1986,192765,2138,2830712,9140
2024,612675,4169,417686,2837,101974,1867,49477,1365,371841,4793,266249,3329,672426,3786,160415,1865,193185,2217,2845928,9357



Total Population (B01003_001E) — MOE not available for controlled estimates


,Alameda,Contra Costa,Marin,Napa,San Francisco,San Mateo,Santa Clara,Solano,Sonoma,Bay Area
Year,,,,,,,,,,
2015,1638215,1126745,261221,142456,864816,765135,1918044,436092,502146,7654870
2016,1647704,1135127,260651,142166,870887,764797,1919402,440207,503070,7684011
2017,1663190,1147439,260955,140973,884363,771410,1938153,445458,504217,7756158
2018,1666753,1150215,259666,139417,883305,769545,1937570,446610,499942,7753023
2019,1671329,1153526,258826,137744,881549,766573,1927852,447643,494336,7739378
2021,1648556,1161413,260206,136207,815201,737888,1885508,451716,485887,7582582
2022,1628997,1156966,256018,134300,808437,729181,1870945,448747,482650,7516241
2023,1622188,1155025,254407,133216,808988,726353,1877592,449218,481812,7508799
2024,1649060,1172607,256400,132727,827526,742893,1926325,455101,485375,7648014



Population in Occupied Housing Units (B25008_001E / B25008_001M)


,Alameda,Alameda MOE,Contra Costa,Contra Costa MOE,Marin,Marin MOE,Napa,Napa MOE,San Francisco,San Francisco MOE,San Mateo,San Mateo MOE,Santa Clara,Santa Clara MOE,Solano,Solano MOE,Sonoma,Sonoma MOE,Bay Area,Bay Area MOE
Year,,,,,,,,,,,,,,,,,,,,
2015,1607666,2302,1116658,1017,253949,1435,138098,977,844811,2897,755303,1372,1881601,2554,425740,1080,493315,1449,7517141,5423
2016,1615422,2872,1125239,1129,253113,1086,137832,987,850083,3368,754699,1319,1883010,2547,429703,1390,494312,1370,7543413,5921
2017,1630189,2704,1137487,1036,252973,1358,137085,798,864051,2827,760952,1718,1902580,2285,434721,1397,496390,1217,7616428,5519
2018,1635704,1803,1140458,944,252051,1604,135617,780,862693,3904,759797,1393,1901147,2925,435966,1258,491395,1340,7614828,6036
2019,1640444,1925,1143695,1043,252218,1787,133806,933,861584,3370,756883,1690,1890540,2684,436472,1222,485679,1428,7601321,5814
2021,1616986,5340,1152343,1084,254712,1489,132827,820,797995,3367,728258,1739,1851492,2526,440441,2575,477828,1550,7452882,7897
2022,1591744,601,1144370,418,248861,227,130255,1228,782233,1609,717387,607,1829890,1186,438087,249,471558,400,7354385,2585
2023,1582968,3873,1141589,240,247462,226,128790,145,778344,910,714286,219,1836523,638,438722,434,470927,205,7339611,4080
2024,1611690,957,1158835,231,249395,200,128737,115,797384,1096,732739,337,1880548,895,444546,311,474717,197,7478591,1809
